### Alle Lösungen finden
Der einfachste Weg ist wahrscheinlich folgender:
1. Finde mit bidirektionaler Suche eine Lösung via einen Mittelpunkt (nicht unbedingt eine kürzeste Lösung)
2. Erstelle mit BFS Distanz-Dicts, die den Bereich vom
   Start zum Mittelpunkt und vom Ziel zum Mittelpunkt abdecken.
3. Finde die gemeinsamen Zustände.
4. Finde die Länge des kürzesten Wegs über diese gemeinsamen Zustände.
   Ist diese Lösung kürzer als die bei 1., passe die Dicts an.
5. Suche vom Mittelpunkt aus kürzeste Pfade zum Start und Ziel.
   Jeder Schritt muss näher zum Start bez. Ziel führen.
   Konsultiere dazu die Distanz-Dicts.

In [ ]:
import babycube as cube
import search_strategies as S
from collections import deque


def get_transition(s, t):
    x = cube.apply_op(t, cube.inv_op(s))
    return cube.OP_KEY.get(x)


def get_neighbors(state):
    for k, op in cube.KEY_OP.items():
        new_state = cube.apply_op(op, state)
        yield new_state


def get_neighbors2(state):
    for k, op in cube.KEY_OP.items():
        new_state = cube.apply_op(op, state)
        yield k, new_state


def pathword(path):
    keys = [get_transition(t, s) for t, s in zip(path[:-1], path[1:])]
    word = ''.join(keys)
    return word


def inv_word(word):
    '''invertiert word, z.B. uR2f -> FR2U'''
    keys = cube.get_keys(word)
    return ''.join(cube.inv_key(k) for k in reversed(keys))


def all_shortest_paths_with_dd(midpoint, dd_start, reverse=False):
    '''dd_start: Distanz-Dict, gibt distanz zu start an, goal muss key von dd_start sein.
       gibt alle kuerzesten Pfade (als Woerter) zurueck, die zum state mit
       dd_start[state] == 0 fuehren.
       reverse=True: gibt die Pfade vom start zum midpoint zurueck.
    '''
    depth = dd_start[midpoint]
    path = ('',)
    nodes_to_visit = deque([(depth, path, midpoint)])

    solutions = []
    while nodes_to_visit:
        depth, path, node = nodes_to_visit.pop()
        if depth == 0:
            solutions.append(''.join(path))
        for k, neighbor in get_neighbors2(node):
            if dd_start.get(neighbor) != depth - 1:  # depth muss abnehmen!
                continue
            new_path = path + (k,) if reverse else (cube.inv_key(k),) + path
            nodes_to_visit.appendleft((depth - 1, new_path, neighbor))

    return solutions


#########


def get_midpoint_and_depths(goal):
    midpoint, go_backs = S.search_bibf(cube.ID, get_neighbors, goal)

    mp_to_start = S.get_path_home(midpoint, go_backs[0])
    mp_to_goal = S.get_path_home(midpoint, go_backs[1])

    return midpoint, len(mp_to_start) - 1, len(mp_to_goal) - 1


def all_paths_via(midpoint, dd_start, dd_goal):
    '''gibt alle kuerzesten Pfade via midpoint zurueck
       dd_start, dd_goal: Dist-Dicts, die einem Zustand die
       Distanz zum start bez. goal zuordnen.
       midpoint ist Key mit maximalem Wert in beiden Dicts
    '''
    paths_to = all_shortest_paths_with_dd(midpoint, dd_start)
    paths_from = all_shortest_paths_with_dd(midpoint, dd_goal, reverse=True)
    all_paths = []
    for p in paths_to:
        for q in paths_from:
            all_paths.append(p+q)
    return all_paths


def all_paths_bi(goal):
    '''gibt alle kuerzesten Pfade zu goal zurueck'''
    paths = []
    midpoint, depth_1, depth_2 = get_midpoint_and_depths(goal)
    if depth_2 == 0:
        if depth_1 == 0:
            return ['']
        return [get_transition(cube.ID, goal)]

    dd_start = S.search_bf(cube.ID, get_neighbors, None, depth_1)[-1]
    dd_goal = S.search_bf(goal, get_neighbors, None, depth_2)[-1]

    midpoints = set(dd_start) & set(dd_goal)
    min_dist = min(dd_start[mp]+dd_goal[mp] for mp in midpoints)

    if min_dist % 2 == 0 and depth_1 > min_dist / 2:
        dd_start = {k: v for k, v in dd_start.items() if v <= min_dist // 2}
        midpoints = set(dd_start) & set(dd_goal)

    for midpoint in midpoints:
        paths = all_paths_via(midpoint, dd_start, dd_goal)
        paths += paths
    return paths

In [ ]:
goal = cube.apply_word('rF2R2fRFUF')
# goal = cube.apply_word('URFU2R2URF2R')
# goal = cube.apply_word('RURURF2RF2RU')
# goal = cube.apply_word('RURURF2RF2R')

In [ ]:
midpoint, go_backs = S.search_bibf(cube.ID, get_neighbors, goal)

mp_to_start = S.get_path_home(midpoint, go_backs[0])
mp_to_goal = S.get_path_home(midpoint, go_backs[1])

depth_1, depth_2 = len(mp_to_start) - 1, len(mp_to_goal) - 1
depth_1, depth_2

In [ ]:
cube.apply_word(pathword(mp_to_goal[::-1]) + pathword(mp_to_start), goal)

In [ ]:
dd_start = S.search_bf(cube.ID, get_neighbors, None, depth_1)[-1]
dd_goal = S.search_bf(goal, get_neighbors, None, depth_2)[-1]

In [ ]:
midpoints = set(dd_start) & set(dd_goal)
len(midpoints)

In [ ]:
min_dist = min(dd_start[mp]+dd_goal[mp] for mp in midpoints)
min_dist

In [ ]:
if min_dist % 2 == 0 and depth_1 > min_dist / 2:
    dd_start = {k: v for k, v in dd_start.items() if v <= min_dist // 2}
    midpoints = set(dd_start) & set(dd_goal)

len(midpoints)

In [ ]:
[all_shortest_paths_with_dd(midpoint, dd_start)
 for midpoint in midpoints
 ]

In [ ]:
[all_shortest_paths_with_dd(midpoint, dd_goal, reverse=True)
 for midpoint in midpoints
 ]

In [ ]:
paths = all_paths_bi(goal)
paths

In [ ]:
{cube.apply_word(word) == goal for word in paths}

In [ ]:
solutions = {inv_word(w) for w in paths}
{cube.apply_word(word, goal) for word in solutions}